# Phase 5: Backtesting, Validation & Research Hygiene  
## Notebook 05_05 — Bayesian Strategy Calibration

### Research question

How do we calibrate strategy parameters while explicitly accounting for uncertainty, prior beliefs, posterior shrinkage, model risk, and the probability that apparent alpha is just noise?

This notebook follows:

```text
05_01_vectorized_backtest_engine
05_02_event_driven_backtest_engine
05_03_transaction_costs_slippage_latency
05_04_walk_forward_optimization
```

The previous notebook selected parameters using walk-forward validation. This notebook adds a Bayesian layer:

> Instead of asking “which parameter set had the best backtest?”, ask “what is the posterior probability that this parameter set has positive risk-adjusted alpha after costs?”

It covers:

1. why point-estimated Sharpe is fragile;
2. Bayesian prior over strategy alpha;
3. likelihood model for strategy returns;
4. conjugate Normal-Inverse-Gamma calibration;
5. posterior alpha and volatility;
6. posterior predictive returns;
7. credible intervals;
8. probability of positive alpha;
9. probability of Sharpe exceeding a threshold;
10. Bayesian shrinkage across parameter sets;
11. posterior model probabilities;
12. Bayesian model averaging;
13. Thompson-style parameter sampling;
14. walk-forward Bayesian updating;
15. posterior predictive drawdown and CVaR;
16. calibration diagnostics;
17. governance flags;
18. saved outputs and manifest.

The key idea:

> A backtest estimate is not a fact. It is noisy evidence that should update a prior into a posterior.

## 1. Why Bayesian strategy calibration?

A parameter grid often produces many strategies.

Some look good by luck.

A frequentist grid search might choose:

$$
\theta^* = \arg\max_\theta \hat{Sharpe}_\theta
$$

But this ignores uncertainty.

Bayesian calibration asks:

$$
p(\alpha_\theta,\sigma_\theta^2 \mid r_{\theta,1:T})
$$

where:

- $\alpha_\theta$ is expected strategy return;
- $\sigma_\theta$ is strategy volatility;
- $r_{\theta,1:T}$ are observed strategy returns.

Then we choose or average strategies using posterior evidence, not just point estimates.

## 2. Bayesian alpha model

For a strategy parameter set $\theta$, suppose daily net returns follow:

$$
r_t \sim N(\mu,\sigma^2)
$$

Use a Normal-Inverse-Gamma prior:

$$
\sigma^2 \sim InvGamma(\alpha_0,\beta_0)
$$

$$
\mu \mid \sigma^2 \sim N\Big(\mu_0,\frac{\sigma^2}{\kappa_0}\Big)
$$

After observing $n$ returns, the posterior is also Normal-Inverse-Gamma:

$$
\kappa_n = \kappa_0+n
$$

$$
\mu_n = \frac{\kappa_0\mu_0+n\bar r}{\kappa_n}
$$

$$
\alpha_n = \alpha_0+\frac{n}{2}
$$

$$
\begin{aligned}
\beta_n &= \beta_0 \\
&\quad + \frac{1}{2}\sum_t(r_t-\bar r)^2 \\
&\quad + \frac{\kappa_0 n(\bar r-\mu_0)^2}{2\kappa_n}
\end{aligned}
$$

This creates shrinkage: noisy strategies are pulled toward the prior.

## 3. Posterior predictive distribution

The posterior predictive distribution accounts for uncertainty in both mean and volatility.

For a new return $r_{new}$, the predictive distribution is Student-t.

This notebook uses Monte Carlo posterior sampling:

1. sample $\sigma^2$ from inverse-gamma posterior;
2. sample $\mu$ conditional on $\sigma^2$;
3. sample future returns from $N(\mu,\sigma^2)$.

This lets us estimate:

- probability of positive alpha;
- probability Sharpe exceeds threshold;
- predictive CVaR;
- predictive drawdown;
- posterior model weights.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class BayesianCalibrationConfig:
    n_days: int = 2_100
    annualisation: int = 252
    seed: int = 42
    train_fraction: float = 0.60
    validation_fraction: float = 0.20
    gross_target: float = 1.20
    max_abs_weight: float = 0.25
    transaction_cost_bps: float = 4.0
    borrow_cost_ann: float = 0.015
    financing_cost_ann: float = 0.035
    prior_mu_ann: float = 0.00
    prior_kappa: float = 20.0
    prior_alpha: float = 4.0
    prior_vol_ann: float = 0.12
    posterior_samples: int = 20_000
    predictive_horizon_days: int = 252
    cvar_alpha: float = 0.95
    sharpe_threshold: float = 0.50
    model_temperature: float = 8.0
    rolling_train_window: int = 504
    rolling_test_window: int = 63
    rolling_step: int = 63

config = BayesianCalibrationConfig()
config

## 5. Simulate multi-asset research data

We simulate a market with weak, regime-dependent signal structure.

The goal is not to produce a magic strategy. The goal is to test whether Bayesian calibration avoids over-trusting noisy winners.

In [ ]:
def simulate_bayesian_research_data(config: BayesianCalibrationConfig):
    rng = np.random.default_rng(config.seed)
    dates = pd.bdate_range("2015-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "EM_EQ", "US_BOND", "EU_BOND",
        "GOLD", "OIL", "COPPER", "FX_CARRY", "TREND", "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity", "EU_EQ": "equity", "EM_EQ": "equity",
        "US_BOND": "bond", "EU_BOND": "bond",
        "GOLD": "commodity", "OIL": "commodity", "COPPER": "commodity",
        "FX_CARRY": "fx", "TREND": "alternative",
        "REIT": "real_asset", "BTC_PROXY": "crypto",
    }

    factor_names = ["market", "rates", "commodity", "carry", "trend", "crypto"]
    factor_vol_daily = np.array([0.010, 0.004, 0.012, 0.006, 0.007, 0.035])
    factor_corr = np.array([
        [ 1.00, -0.25,  0.25,  0.15, -0.15,  0.35],
        [-0.25,  1.00,  0.10, -0.05,  0.15, -0.20],
        [ 0.25,  0.10,  1.00,  0.10,  0.05,  0.20],
        [ 0.15, -0.05,  0.10,  1.00,  0.00,  0.15],
        [-0.15,  0.15,  0.05,  0.00,  1.00,  0.00],
        [ 0.35, -0.20,  0.20,  0.15,  0.00,  1.00],
    ])
    factor_cov = np.outer(factor_vol_daily, factor_vol_daily) * factor_corr

    loadings = pd.DataFrame(0.0, index=assets, columns=factor_names)
    loadings.loc[["US_EQ", "EU_EQ", "EM_EQ", "REIT"], "market"] = [1.00, 0.95, 1.20, 0.70]
    loadings.loc[["US_BOND", "EU_BOND"], "rates"] = [1.00, 0.90]
    loadings.loc[["GOLD", "OIL", "COPPER"], "commodity"] = [0.45, 1.10, 0.95]
    loadings.loc["FX_CARRY", "carry"] = 1.00
    loadings.loc["TREND", "trend"] = 1.00
    loadings.loc["TREND", "market"] = -0.25
    loadings.loc["BTC_PROXY", "crypto"] = 1.00
    loadings.loc["BTC_PROXY", "market"] = 0.40
    loadings.loc["GOLD", "rates"] = 0.25
    loadings.loc["REIT", "rates"] = -0.30

    ann_mu = pd.Series({
        "US_EQ": 0.070, "EU_EQ": 0.060, "EM_EQ": 0.080,
        "US_BOND": 0.025, "EU_BOND": 0.020,
        "GOLD": 0.035, "OIL": 0.045, "COPPER": 0.040,
        "FX_CARRY": 0.030, "TREND": 0.050, "REIT": 0.060, "BTC_PROXY": 0.120,
    })

    idio_vol_ann = pd.Series({
        "US_EQ": 0.07, "EU_EQ": 0.08, "EM_EQ": 0.13,
        "US_BOND": 0.025, "EU_BOND": 0.030,
        "GOLD": 0.12, "OIL": 0.22, "COPPER": 0.18,
        "FX_CARRY": 0.07, "TREND": 0.08, "REIT": 0.11, "BTC_PROXY": 0.52,
    })

    regime = np.zeros(config.n_days, dtype=int)
    stress_type = np.array(["normal"] * config.n_days, dtype=object)
    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    returns = np.zeros((config.n_days, len(assets)))
    factor_returns = np.zeros((config.n_days, len(factor_names)))

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.4
        f = rng.multivariate_normal(np.zeros(len(factor_names)), factor_cov * vol_multiplier**2)

        u = rng.uniform()
        if u < 0.008:
            stress_type[t] = "equity_crash"
            f[0] -= rng.uniform(0.035, 0.100)
            f[1] += rng.uniform(0.004, 0.025)
            f[5] -= rng.uniform(0.080, 0.220)
            f[4] += rng.uniform(0.005, 0.040)
        elif u < 0.014:
            stress_type[t] = "inflation_shock"
            f[1] -= rng.uniform(0.010, 0.035)
            f[2] += rng.uniform(0.030, 0.090)
            f[0] -= rng.uniform(0.010, 0.050)
        elif u < 0.020:
            stress_type[t] = "commodity_crash"
            f[2] -= rng.uniform(0.040, 0.120)
            f[0] -= rng.uniform(0.005, 0.030)

        eps = rng.standard_t(df=6, size=len(assets)) * np.sqrt((6 - 2) / 6)
        eps = eps * idio_vol_ann.values / np.sqrt(config.annualisation) * vol_multiplier * 0.35

        predictive_tilt = np.zeros(len(assets))
        if t > 21:
            recent_market = returns[t-21:t, assets.index("US_EQ")].mean()
            recent_commodity = returns[t-21:t, assets.index("OIL")].mean()
            if recent_market > 0:
                predictive_tilt[assets.index("US_EQ")] += 0.00015
                predictive_tilt[assets.index("EU_EQ")] += 0.00010
            if recent_commodity < 0:
                predictive_tilt[assets.index("GOLD")] += 0.00010
            if regime[t - 1] == 1:
                predictive_tilt[assets.index("TREND")] += 0.00055
                predictive_tilt[assets.index("FX_CARRY")] -= 0.00035

        returns[t] = ann_mu.values / config.annualisation + loadings.to_numpy() @ f + eps + predictive_tilt
        factor_returns[t] = f

    returns = pd.DataFrame(returns, index=dates, columns=assets)
    prices = 100 * np.exp(np.log1p(returns).cumsum())

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "annual_mean_assumption": [ann_mu[a] for a in assets],
        "idio_vol_ann": [idio_vol_ann[a] for a in assets],
    })

    factors = pd.DataFrame(factor_returns, index=dates, columns=factor_names)
    regime_info = pd.DataFrame({"regime": regime, "stress_type": stress_type}, index=dates)
    benchmark = 0.60 * returns["US_EQ"] + 0.25 * returns["US_BOND"] + 0.15 * returns["GOLD"]

    return returns, prices, metadata, factors, regime_info, benchmark

returns, prices, metadata, factors, regime_info, benchmark = simulate_bayesian_research_data(config)
assets = metadata["asset"].tolist()

returns.head(), metadata.head()

In [ ]:
plt.figure(figsize=(12, 6))
for asset in assets:
    plt.plot(prices.index, prices[asset], label=asset, alpha=0.75)
plt.title("Synthetic Prices")
plt.xlabel("Date")
plt.ylabel("Price index")
plt.legend(ncol=3)
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(regime_info.index, regime_info["regime"], drawstyle="steps-post")
plt.title("Regime Indicator")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.yticks([0, 1])
plt.show()

## 6. Parameterised strategy family

Each strategy variant has:

- momentum lookback;
- reversal lookback;
- volatility lookback;
- momentum weight;
- reversal weight;
- volatility penalty weight;
- rebalance frequency.

The Bayesian calibration treats each parameter set as a model.

In [ ]:
def cross_sectional_zscore(df):
    mu = df.mean(axis=1)
    sigma = df.std(axis=1).replace(0.0, np.nan)
    return df.sub(mu, axis=0).div(sigma, axis=0).fillna(0.0)

def make_signal(prices, returns, params, annualisation):
    momentum = prices.pct_change(params["momentum_lookback"])
    reversal = -prices.pct_change(params["reversal_lookback"])
    vol = returns.rolling(params["vol_lookback"]).std() * np.sqrt(annualisation)

    signal = (
        params["momentum_weight"] * cross_sectional_zscore(momentum)
        + params["reversal_weight"] * cross_sectional_zscore(reversal)
        + params["vol_penalty_weight"] * cross_sectional_zscore(-vol)
    )
    return signal.clip(-3, 3).fillna(0.0)

def signal_to_weights(signal, returns, config, params):
    vol = returns.rolling(params["vol_lookback"]).std().shift(1)
    vol = vol.fillna(returns.expanding().std().shift(1)).fillna(returns.std())
    inv_vol = 1.0 / vol.replace(0.0, np.nan)

    raw = signal.sub(signal.mean(axis=1), axis=0).mul(inv_vol, axis=0)
    weights = raw.div(raw.abs().sum(axis=1).replace(0.0, np.nan), axis=0).mul(config.gross_target).fillna(0.0)
    weights = weights.clip(-config.max_abs_weight, config.max_abs_weight)

    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    weights = weights.div(gross, axis=0).mul(config.gross_target).fillna(0.0)
    return weights

def apply_rebalance(weights, step):
    scheduled = weights.copy() * np.nan
    scheduled.iloc[::step] = weights.iloc[::step]
    return scheduled.ffill().fillna(0.0)

def params_to_id(params):
    return (
        f"mom{params['momentum_lookback']}_"
        f"rev{params['reversal_lookback']}_"
        f"vol{params['vol_lookback']}_"
        f"mw{params['momentum_weight']}_"
        f"rw{params['reversal_weight']}_"
        f"vw{params['vol_penalty_weight']}_"
        f"reb{params['rebalance_step']}"
    )

def make_parameter_grid():
    grid = []
    for momentum_lookback in [21, 63, 126]:
        for reversal_lookback in [5, 10, 21]:
            for vol_lookback in [21, 63]:
                for momentum_weight in [0.4, 0.7]:
                    for reversal_weight in [0.1, 0.3]:
                        for vol_penalty_weight in [0.1, 0.3]:
                            for rebalance_step in [5, 10, 21]:
                                grid.append({
                                    "momentum_lookback": momentum_lookback,
                                    "reversal_lookback": reversal_lookback,
                                    "vol_lookback": vol_lookback,
                                    "momentum_weight": momentum_weight,
                                    "reversal_weight": reversal_weight,
                                    "vol_penalty_weight": vol_penalty_weight,
                                    "rebalance_step": rebalance_step,
                                })
    return grid

param_grid = make_parameter_grid()
pd.Series({"n_parameter_sets": len(param_grid), "example": str(param_grid[0])})

## 7. Backtest each parameter set

We generate net returns for every parameter set.

The same cost model is applied to all variants.

In [ ]:
def run_strategy_returns(prices, returns, params, config):
    signal = make_signal(prices, returns, params, config.annualisation)
    target_weights = signal_to_weights(signal, returns, config, params)
    target_weights = apply_rebalance(target_weights, params["rebalance_step"])
    held = target_weights.shift(1).fillna(0.0)

    gross = (held * returns).sum(axis=1)

    delta = held.diff().fillna(held)
    turnover = delta.abs().sum(axis=1)
    transaction_cost = turnover * config.transaction_cost_bps / 10000.0

    short_exposure = held.clip(upper=0.0).abs().sum(axis=1)
    borrow_cost = short_exposure * config.borrow_cost_ann / config.annualisation

    gross_exposure = held.abs().sum(axis=1)
    financing_cost = np.maximum(gross_exposure - 1.0, 0.0) * config.financing_cost_ann / config.annualisation

    net = gross - transaction_cost - borrow_cost - financing_cost

    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": transaction_cost,
        "borrow_cost": borrow_cost,
        "financing_cost": financing_cost,
        "net_return": net,
        "turnover": turnover,
        "gross_exposure": gross_exposure,
        "net_exposure": held.sum(axis=1),
    }, index=returns.index), target_weights

strategy_return_frames = []
strategy_metric_rows = []
strategy_weight_examples = {}

for params in param_grid:
    param_id = params_to_id(params)
    bt, weights = run_strategy_returns(prices, returns, params, config)
    strategy_return_frames.append(bt[["net_return"]].rename(columns={"net_return": param_id}))
    strategy_weight_examples[param_id] = weights

strategy_returns = pd.concat(strategy_return_frames, axis=1).fillna(0.0)

strategy_returns.head()

## 8. Train / validation / test split

Bayesian calibration estimates posterior beliefs from training data.

Validation checks whether posterior ranking helps select reasonable strategies.

Test is held out for final evaluation.

In [ ]:
n = len(strategy_returns)
train_end = int(n * config.train_fraction)
validation_end = int(n * (config.train_fraction + config.validation_fraction))

train_returns = strategy_returns.iloc[:train_end]
validation_returns = strategy_returns.iloc[train_end:validation_end]
test_returns = strategy_returns.iloc[validation_end:]

split_report = pd.Series({
    "n_total_days": n,
    "n_train_days": len(train_returns),
    "n_validation_days": len(validation_returns),
    "n_test_days": len(test_returns),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "validation_start": str(validation_returns.index[0]),
    "validation_end": str(validation_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
})

split_report

## 9. Frequentist point estimates

Before Bayesian calibration, inspect naive estimates:

$$
\hat{\mu}, \hat{\sigma}, \widehat{Sharpe}
$$

The best point-estimate Sharpe is usually optimistic.

In [ ]:
def max_drawdown(nav):
    dd = nav / nav.cummax() - 1.0
    return dd, float(dd.min())

def historical_var_cvar(losses, alpha):
    var = np.quantile(losses, alpha)
    tail = losses[losses >= var]
    return float(var), float(tail.mean()) if len(tail) else np.nan

def strategy_metrics(return_series, config):
    r = return_series.dropna()
    if len(r) == 0:
        return {}

    nav = (1 + r).cumprod()
    _, mdd = max_drawdown(nav)
    ann_return = (1 + r).prod() ** (config.annualisation / len(r)) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(config.annualisation)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    var, cvar = historical_var_cvar(-r, config.cvar_alpha)

    return {
        "n_obs": len(r),
        "mean_daily": float(r.mean()),
        "vol_daily": float(r.std(ddof=1)),
        "annualised_return": float(ann_return),
        "annualised_vol": float(ann_vol),
        "sharpe_proxy": float(sharpe),
        "max_drawdown": float(mdd),
        "VaR": var,
        "CVaR": cvar,
        "total_return": float(nav.iloc[-1] - 1),
    }

point_rows = []
for param_id in strategy_returns.columns:
    m = strategy_metrics(train_returns[param_id], config)
    point_rows.append({"param_id": param_id, **m})

point_estimates_train = pd.DataFrame(point_rows).sort_values("sharpe_proxy", ascending=False)

point_estimates_train.head(10)

## 10. Normal-Inverse-Gamma posterior

We calibrate posterior parameters for each strategy:

$$
(\mu_n,\kappa_n,\alpha_n,\beta_n)
$$

The prior mean is conservative:

$$
\mu_0 = 0
$$

with prior strength $\kappa_0$.

In [ ]:
def nig_prior_from_config(config):
    mu0 = config.prior_mu_ann / config.annualisation
    kappa0 = config.prior_kappa
    alpha0 = config.prior_alpha

    prior_vol_daily = config.prior_vol_ann / np.sqrt(config.annualisation)
    # For InvGamma(alpha, beta), mean variance is beta / (alpha - 1)
    beta0 = prior_vol_daily**2 * (alpha0 - 1)

    return mu0, kappa0, alpha0, beta0

def fit_normal_inverse_gamma(returns_series, config):
    r = returns_series.dropna().to_numpy()
    n = len(r)

    mu0, kappa0, alpha0, beta0 = nig_prior_from_config(config)

    if n == 0:
        return {
            "n": 0,
            "sample_mean": np.nan,
            "sample_var": np.nan,
            "mu_n": mu0,
            "kappa_n": kappa0,
            "alpha_n": alpha0,
            "beta_n": beta0,
        }

    sample_mean = r.mean()
    sample_var = r.var(ddof=1) if n > 1 else 0.0
    sse = ((r - sample_mean) ** 2).sum()

    kappa_n = kappa0 + n
    mu_n = (kappa0 * mu0 + n * sample_mean) / kappa_n
    alpha_n = alpha0 + n / 2.0
    beta_n = beta0 + 0.5 * sse + (kappa0 * n * (sample_mean - mu0) ** 2) / (2.0 * kappa_n)

    return {
        "n": n,
        "sample_mean": sample_mean,
        "sample_var": sample_var,
        "mu_n": mu_n,
        "kappa_n": kappa_n,
        "alpha_n": alpha_n,
        "beta_n": beta_n,
    }

posterior_rows = []
for param_id in train_returns.columns:
    post = fit_normal_inverse_gamma(train_returns[param_id], config)
    posterior_rows.append({"param_id": param_id, **post})

posterior_table = pd.DataFrame(posterior_rows)

posterior_table.head()

## 11. Posterior sampling

We sample:

$$
\sigma^2 \sim InvGamma(\alpha_n,\beta_n)
$$

$$
\mu \mid \sigma^2 \sim N(\mu_n,\sigma^2/\kappa_n)
$$

Then compute posterior Sharpe samples:

$$
Sharpe = \frac{\mu \times 252}{\sigma \sqrt{252}}
$$

In [ ]:
def sample_inverse_gamma(alpha, beta, size, rng):
    # If X ~ Gamma(alpha, scale=1/beta), then 1/X ~ InvGamma(alpha, beta)
    gamma_samples = rng.gamma(shape=alpha, scale=1.0 / beta, size=size)
    return 1.0 / gamma_samples

def sample_posterior_mu_sigma(post, config, rng, n_samples=None):
    if n_samples is None:
        n_samples = config.posterior_samples

    sigma2 = sample_inverse_gamma(post["alpha_n"], post["beta_n"], n_samples, rng)
    mu = rng.normal(loc=post["mu_n"], scale=np.sqrt(sigma2 / post["kappa_n"]))

    sigma = np.sqrt(sigma2)
    sharpe = (mu * config.annualisation) / (sigma * np.sqrt(config.annualisation))

    return mu, sigma, sharpe

rng = np.random.default_rng(config.seed + 1000)

posterior_summary_rows = []
posterior_sample_store = {}

for _, row in posterior_table.iterrows():
    post = row.to_dict()
    param_id = post["param_id"]

    mu_s, sigma_s, sharpe_s = sample_posterior_mu_sigma(post, config, rng)
    posterior_sample_store[param_id] = {
        "mu": mu_s,
        "sigma": sigma_s,
        "sharpe": sharpe_s,
    }

    posterior_summary_rows.append({
        "param_id": param_id,
        "posterior_mu_ann_mean": mu_s.mean() * config.annualisation,
        "posterior_mu_ann_p05": np.quantile(mu_s * config.annualisation, 0.05),
        "posterior_mu_ann_p50": np.quantile(mu_s * config.annualisation, 0.50),
        "posterior_mu_ann_p95": np.quantile(mu_s * config.annualisation, 0.95),
        "posterior_vol_ann_mean": sigma_s.mean() * np.sqrt(config.annualisation),
        "posterior_sharpe_mean": sharpe_s.mean(),
        "posterior_sharpe_p05": np.quantile(sharpe_s, 0.05),
        "posterior_sharpe_p50": np.quantile(sharpe_s, 0.50),
        "posterior_sharpe_p95": np.quantile(sharpe_s, 0.95),
        "prob_mu_positive": np.mean(mu_s > 0),
        "prob_sharpe_above_threshold": np.mean(sharpe_s > config.sharpe_threshold),
    })

posterior_summary = pd.DataFrame(posterior_summary_rows).sort_values("posterior_sharpe_mean", ascending=False)

posterior_summary.head(10)

In [ ]:
top_param = posterior_summary.iloc[0]["param_id"]
top_samples = posterior_sample_store[top_param]

plt.figure(figsize=(10, 6))
plt.hist(top_samples["sharpe"], bins=80, density=True)
plt.axvline(config.sharpe_threshold, linestyle="--", label=f"Sharpe threshold {config.sharpe_threshold}")
plt.title(f"Posterior Sharpe Distribution: {top_param}")
plt.xlabel("Sharpe")
plt.ylabel("Density")
plt.legend()
plt.show()

## 12. Bayesian shrinkage versus point estimates

The posterior mean alpha is shrunk toward the prior.

Noisy high-Sharpe strategies should lose some of their shine.

In [ ]:
shrinkage_comparison = (
    point_estimates_train[["param_id", "annualised_return", "annualised_vol", "sharpe_proxy"]]
    .merge(
        posterior_summary[[
            "param_id",
            "posterior_mu_ann_mean",
            "posterior_sharpe_mean",
            "prob_mu_positive",
            "prob_sharpe_above_threshold",
        ]],
        on="param_id",
        how="left"
    )
)

shrinkage_comparison["return_shrinkage"] = shrinkage_comparison["posterior_mu_ann_mean"] - shrinkage_comparison["annualised_return"]
shrinkage_comparison["sharpe_shrinkage"] = shrinkage_comparison["posterior_sharpe_mean"] - shrinkage_comparison["sharpe_proxy"]

shrinkage_comparison.sort_values("sharpe_proxy", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(shrinkage_comparison["sharpe_proxy"], shrinkage_comparison["posterior_sharpe_mean"], alpha=0.7)
plt.axline((0, 0), slope=1, linestyle="--")
plt.title("Point Estimate Sharpe vs Posterior Mean Sharpe")
plt.xlabel("Training point-estimate Sharpe")
plt.ylabel("Posterior mean Sharpe")
plt.show()

## 13. Posterior model probabilities

We convert posterior evidence into model weights.

A simple softmax over posterior expected utility:

$$
p(M_i \mid data) \propto \exp(\tau \cdot score_i)
$$

where score can be posterior mean Sharpe adjusted by tail and uncertainty.

This is not a full marginal likelihood calculation, but it is a practical Bayesian-style model weighting scheme.

In [ ]:
def model_probability_score(row):
    # Reward posterior Sharpe and confidence; penalise uncertainty.
    uncertainty_penalty = row["posterior_sharpe_p95"] - row["posterior_sharpe_p05"]
    score = (
        row["posterior_sharpe_mean"]
        + 0.75 * row["prob_sharpe_above_threshold"]
        - 0.10 * uncertainty_penalty
    )
    return score

model_weights = posterior_summary.copy()
model_weights["model_score"] = model_weights.apply(model_probability_score, axis=1)

score = model_weights["model_score"].to_numpy()
score = score - score.max()
raw_prob = np.exp(config.model_temperature * score)
model_weights["posterior_model_probability"] = raw_prob / raw_prob.sum()

model_weights = model_weights.sort_values("posterior_model_probability", ascending=False)

model_weights.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plot_df = model_weights.head(15).sort_values("posterior_model_probability")
plt.barh(plot_df["param_id"], plot_df["posterior_model_probability"])
plt.title("Top Posterior Model Probabilities")
plt.xlabel("Probability")
plt.ylabel("Parameter set")
plt.show()

## 14. Bayesian model averaging

Instead of selecting one parameter set, combine strategy returns:

$$
r^{BMA}_t = \sum_i p_i r_{i,t}
$$

where:

$$
p_i = p(M_i \mid data)
$$

This diversifies parameter uncertainty.

In [ ]:
prob_series = model_weights.set_index("param_id")["posterior_model_probability"].reindex(strategy_returns.columns).fillna(0.0)
prob_series = prob_series / prob_series.sum()

bma_returns = strategy_returns @ prob_series

top_selected_param = model_weights.iloc[0]["param_id"]
point_best_param = point_estimates_train.iloc[0]["param_id"]

comparison_returns = pd.DataFrame({
    "bayesian_model_average": bma_returns,
    "posterior_top_model": strategy_returns[top_selected_param],
    "point_estimate_best_train": strategy_returns[point_best_param],
    "equal_model_weight": strategy_returns.mean(axis=1),
})

comparison_returns.head()

## 15. Validation and test performance of Bayesian choices

We compare:

1. Bayesian model average;
2. top posterior model;
3. point-estimate best model;
4. equal model-weight ensemble.

The key comparison is validation and test, not train.

In [ ]:
def performance_table_for_returns(return_frame, config, label_prefix=""):
    rows = []
    for col in return_frame.columns:
        for period_name, period_returns in [
            ("train", return_frame[col].iloc[:train_end]),
            ("validation", return_frame[col].iloc[train_end:validation_end]),
            ("test", return_frame[col].iloc[validation_end:]),
        ]:
            metrics = strategy_metrics(period_returns, config)
            rows.append({
                "strategy": col,
                "period": period_name,
                **metrics,
            })
    return pd.DataFrame(rows)

bayesian_choice_performance = performance_table_for_returns(comparison_returns, config)

bayesian_choice_performance

In [ ]:
plt.figure(figsize=(12, 6))
for col in comparison_returns.columns:
    nav = (1 + comparison_returns[col].iloc[validation_end:]).cumprod()
    plt.plot(nav.index, nav, label=col)
plt.title("Test-Period Performance of Bayesian Calibration Choices")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 16. Posterior predictive simulation

For selected strategies, simulate future one-year returns from posterior samples.

This estimates predictive uncertainty in:

- terminal return;
- max drawdown;
- CVaR;
- probability of loss.

In [ ]:
def simulate_posterior_predictive(param_id, posterior_sample_store, config, rng):
    samples = posterior_sample_store[param_id]
    n_samples = len(samples["mu"])
    horizon = config.predictive_horizon_days

    idx = rng.integers(0, n_samples, size=n_samples)
    mu = samples["mu"][idx]
    sigma = samples["sigma"][idx]

    predictive_returns = rng.normal(
        loc=mu[:, None],
        scale=sigma[:, None],
        size=(n_samples, horizon)
    )

    terminal = np.prod(1 + predictive_returns, axis=1) - 1
    paths = np.cumprod(1 + predictive_returns, axis=1)
    running_max = np.maximum.accumulate(paths, axis=1)
    drawdowns = paths / running_max - 1
    max_dd = drawdowns.min(axis=1)

    daily_losses = -predictive_returns.mean(axis=1)
    terminal_losses = -terminal
    var, cvar = historical_var_cvar(terminal_losses, config.cvar_alpha)

    return {
        "terminal_return": terminal,
        "max_drawdown": max_dd,
        "terminal_loss_VaR": var,
        "terminal_loss_CVaR": cvar,
        "prob_terminal_loss": np.mean(terminal < 0),
        "predictive_paths": paths,
    }

rng_pred = np.random.default_rng(config.seed + 2222)
predictive_results = {}

for param_id in [top_selected_param, point_best_param]:
    predictive_results[param_id] = simulate_posterior_predictive(param_id, posterior_sample_store, config, rng_pred)

predictive_summary = pd.DataFrame([
    {
        "param_id": param_id,
        "mean_terminal_return": result["terminal_return"].mean(),
        "median_terminal_return": np.median(result["terminal_return"]),
        "p05_terminal_return": np.quantile(result["terminal_return"], 0.05),
        "p95_terminal_return": np.quantile(result["terminal_return"], 0.95),
        "mean_max_drawdown": result["max_drawdown"].mean(),
        "p05_max_drawdown": np.quantile(result["max_drawdown"], 0.05),
        "prob_terminal_loss": result["prob_terminal_loss"],
        "terminal_loss_VaR": result["terminal_loss_VaR"],
        "terminal_loss_CVaR": result["terminal_loss_CVaR"],
    }
    for param_id, result in predictive_results.items()
])

predictive_summary

In [ ]:
plt.figure(figsize=(10, 6))
for param_id, result in predictive_results.items():
    plt.hist(result["terminal_return"], bins=80, alpha=0.5, density=True, label=param_id)
plt.axvline(0, linestyle="--")
plt.title("Posterior Predictive One-Year Terminal Return")
plt.xlabel("Terminal return")
plt.ylabel("Density")
plt.legend()
plt.show()

## 17. Thompson-style parameter sampling

A Bayesian decision process can sample parameter sets according to posterior model probability.

This avoids always choosing the current top model and naturally explores uncertainty.

Here we generate simulated parameter selections from posterior model probabilities.

In [ ]:
rng_thompson = np.random.default_rng(config.seed + 3333)

thompson_samples = rng_thompson.choice(
    prob_series.index,
    size=10_000,
    replace=True,
    p=prob_series.values
)

thompson_selection_frequency = (
    pd.Series(thompson_samples)
    .value_counts(normalize=True)
    .rename_axis("param_id")
    .reset_index(name="sample_frequency")
    .merge(model_weights[["param_id", "posterior_model_probability"]], on="param_id", how="left")
    .sort_values("sample_frequency", ascending=False)
)

thompson_selection_frequency.head(10)

## 18. Walk-forward Bayesian updating

Now we run a walk-forward process:

1. use trailing training window;
2. fit Bayesian posterior for each parameter set;
3. compute posterior model probabilities;
4. trade Bayesian model average on the next test window;
5. roll forward.

This mimics an online research process.

In [ ]:
def make_walk_forward_splits(index, train_window, test_window, step):
    rows = []
    start = train_window
    while start + test_window <= len(index):
        rows.append({
            "split_id": len(rows),
            "train_start_idx": start - train_window,
            "train_end_idx": start,
            "test_start_idx": start,
            "test_end_idx": start + test_window,
            "train_start": index[start - train_window],
            "train_end": index[start - 1],
            "test_start": index[start],
            "test_end": index[start + test_window - 1],
        })
        start += step
    return pd.DataFrame(rows)

wf_splits = make_walk_forward_splits(
    strategy_returns.index,
    config.rolling_train_window,
    config.rolling_test_window,
    config.rolling_step,
)

wf_splits.head()

In [ ]:
def posterior_model_weights_from_window(train_window_returns, config):
    rows = []
    for param_id in train_window_returns.columns:
        post = fit_normal_inverse_gamma(train_window_returns[param_id], config)
        rng_local = np.random.default_rng(abs(hash(param_id)) % (2**32))
        mu_s, sigma_s, sharpe_s = sample_posterior_mu_sigma(post, config, rng_local, n_samples=5_000)

        rows.append({
            "param_id": param_id,
            "posterior_mu_ann_mean": mu_s.mean() * config.annualisation,
            "posterior_sharpe_mean": sharpe_s.mean(),
            "posterior_sharpe_p05": np.quantile(sharpe_s, 0.05),
            "posterior_sharpe_p95": np.quantile(sharpe_s, 0.95),
            "prob_mu_positive": np.mean(mu_s > 0),
            "prob_sharpe_above_threshold": np.mean(sharpe_s > config.sharpe_threshold),
        })

    table = pd.DataFrame(rows)
    table["model_score"] = table.apply(model_probability_score, axis=1)
    score = table["model_score"].to_numpy()
    score = score - score.max()
    raw = np.exp(config.model_temperature * score)
    table["posterior_model_probability"] = raw / raw.sum()

    return table.sort_values("posterior_model_probability", ascending=False)

wf_return_frames = []
wf_weight_frames = []
wf_model_weight_frames = []
wf_selected_rows = []

for _, split in wf_splits.iterrows():
    train_window = strategy_returns.iloc[int(split["train_start_idx"]):int(split["train_end_idx"])]
    test_window = strategy_returns.iloc[int(split["test_start_idx"]):int(split["test_end_idx"])]

    mw = posterior_model_weights_from_window(train_window, config)
    probs = mw.set_index("param_id")["posterior_model_probability"].reindex(strategy_returns.columns).fillna(0.0)
    probs = probs / probs.sum()

    bma_test = test_window @ probs
    top_param = mw.iloc[0]["param_id"]
    top_test = test_window[top_param]

    split_returns = pd.DataFrame({
        "bayesian_model_average": bma_test,
        "top_posterior_model": top_test,
    }, index=test_window.index)
    split_returns["split_id"] = int(split["split_id"])
    wf_return_frames.append(split_returns)

    mw["split_id"] = int(split["split_id"])
    wf_model_weight_frames.append(mw)

    wf_selected_rows.append({
        "split_id": int(split["split_id"]),
        "top_param_id": top_param,
        "top_model_probability": mw.iloc[0]["posterior_model_probability"],
        "effective_n_models": 1.0 / np.sum(mw["posterior_model_probability"].to_numpy() ** 2),
        "test_start": split["test_start"],
        "test_end": split["test_end"],
    })

wf_bayesian_returns = pd.concat(wf_return_frames).sort_index()
wf_model_weights = pd.concat(wf_model_weight_frames, ignore_index=True)
wf_selected_summary = pd.DataFrame(wf_selected_rows)

wf_bayesian_returns.head(), wf_selected_summary.head()

## 19. Walk-forward Bayesian performance

In [ ]:
wf_bayesian_performance = performance_table_for_returns(wf_bayesian_returns[["bayesian_model_average", "top_posterior_model"]], config)
wf_bayesian_test_performance = wf_bayesian_performance[wf_bayesian_performance["period"] == "test"].copy()

# Since wf_bayesian_returns is already a stitched OOS series, compute direct metrics too.
wf_direct_rows = []
for col in ["bayesian_model_average", "top_posterior_model"]:
    wf_direct_rows.append({"strategy": f"walk_forward_{col}", **strategy_metrics(wf_bayesian_returns[col], config)})

wf_direct_performance = pd.DataFrame(wf_direct_rows)

wf_direct_performance

In [ ]:
plt.figure(figsize=(12, 6))
for col in ["bayesian_model_average", "top_posterior_model"]:
    nav = (1 + wf_bayesian_returns[col]).cumprod()
    plt.plot(nav.index, nav, label=col)
plt.title("Walk-Forward Bayesian Calibration")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

## 20. Posterior model concentration over time

If posterior probability collapses into one unstable model, that is a governance concern.

We inspect:

$$
N_{eff}=\frac{1}{\sum_i p_i^2}
$$

where $p_i$ are posterior model probabilities.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(wf_selected_summary["split_id"], wf_selected_summary["effective_n_models"], marker="o")
plt.title("Effective Number of Posterior Models Through Time")
plt.xlabel("Split")
plt.ylabel("Effective number of models")
plt.show()

wf_selected_summary.head()

## 21. Regime-conditioned Bayesian performance

Bayesian calibration should be checked in stress regimes.

In [ ]:
def regime_performance(return_series, regime_info, config):
    joined = pd.concat([return_series.rename("return"), regime_info], axis=1).dropna()
    rows = []

    for label, group in joined.groupby("stress_type"):
        if len(group) < 5:
            continue
        rows.append({
            "bucket": label,
            "n_days": len(group),
            **strategy_metrics(group["return"], config),
        })

    for label, group in joined.groupby("regime"):
        if len(group) < 5:
            continue
        rows.append({
            "bucket": f"regime_{label}",
            "n_days": len(group),
            **strategy_metrics(group["return"], config),
        })

    return pd.DataFrame(rows)

bayesian_regime_perf = regime_performance(wf_bayesian_returns["bayesian_model_average"], regime_info, config)

bayesian_regime_perf

## 22. Calibration diagnostics

A Bayesian model should be calibrated, not merely confident.

We check whether realised test returns fall inside posterior predictive intervals.

For simplicity, we evaluate the top posterior model from the initial training sample on the test period.

In [ ]:
def posterior_predictive_interval_for_returns(param_id, posterior_sample_store, config, n_draws):
    rng = np.random.default_rng(config.seed + 4444)
    samples = posterior_sample_store[param_id]
    idx = rng.integers(0, len(samples["mu"]), size=n_draws)
    mu = samples["mu"][idx]
    sigma = samples["sigma"][idx]
    draws = rng.normal(mu, sigma)
    return {
        "p01": np.quantile(draws, 0.01),
        "p05": np.quantile(draws, 0.05),
        "p50": np.quantile(draws, 0.50),
        "p95": np.quantile(draws, 0.95),
        "p99": np.quantile(draws, 0.99),
    }

interval = posterior_predictive_interval_for_returns(top_selected_param, posterior_sample_store, config, config.posterior_samples)
test_series_top = test_returns[top_selected_param]

calibration_report = pd.DataFrame([{
    "param_id": top_selected_param,
    **interval,
    "test_below_p05_rate": float((test_series_top < interval["p05"]).mean()),
    "test_above_p95_rate": float((test_series_top > interval["p95"]).mean()),
    "test_inside_90pct_interval_rate": float(((test_series_top >= interval["p05"]) & (test_series_top <= interval["p95"])).mean()),
    "expected_inside_90pct_interval_rate": 0.90,
}])

calibration_report

## 23. Governance flags

A Bayesian calibration process should be reviewed if:

1. posterior confidence is too concentrated;
2. top model has low probability of positive alpha;
3. posterior predictive intervals are miscalibrated;
4. Bayesian model average underperforms simple alternatives;
5. point-estimate best strongly beats posterior selection only in-sample;
6. stress-regime performance is poor.

In [ ]:
bma_test_metrics = bayesian_choice_performance[
    (bayesian_choice_performance["strategy"] == "bayesian_model_average")
    & (bayesian_choice_performance["period"] == "test")
].iloc[0]

point_test_metrics = bayesian_choice_performance[
    (bayesian_choice_performance["strategy"] == "point_estimate_best_train")
    & (bayesian_choice_performance["period"] == "test")
].iloc[0]

top_model_prob = model_weights.iloc[0]["posterior_model_probability"]
top_prob_mu_positive = model_weights.iloc[0]["prob_mu_positive"]
effective_n_models = 1.0 / np.sum(model_weights["posterior_model_probability"].to_numpy() ** 2)
inside_rate = calibration_report["test_inside_90pct_interval_rate"].iloc[0]
worst_regime_dd = bayesian_regime_perf["max_drawdown"].min() if len(bayesian_regime_perf) else np.nan

governance_flags = pd.DataFrame([{
    "top_model_probability": top_model_prob,
    "effective_n_models": effective_n_models,
    "top_model_prob_mu_positive": top_prob_mu_positive,
    "bma_test_sharpe": bma_test_metrics["sharpe_proxy"],
    "point_best_test_sharpe": point_test_metrics["sharpe_proxy"],
    "predictive_90pct_interval_realised_coverage": inside_rate,
    "worst_regime_drawdown": worst_regime_dd,
    "wf_bma_sharpe": wf_direct_performance[wf_direct_performance["strategy"] == "walk_forward_bayesian_model_average"]["sharpe_proxy"].iloc[0],
    "wf_top_model_sharpe": wf_direct_performance[wf_direct_performance["strategy"] == "walk_forward_top_posterior_model"]["sharpe_proxy"].iloc[0],
    "flag_top_model_too_dominant": bool(top_model_prob > 0.75),
    "flag_top_model_low_positive_alpha_probability": bool(top_prob_mu_positive < 0.60),
    "flag_effective_n_models_below_2": bool(effective_n_models < 2),
    "flag_predictive_interval_miscalibrated": bool(abs(inside_rate - 0.90) > 0.15),
    "flag_bma_test_sharpe_negative": bool(bma_test_metrics["sharpe_proxy"] < 0),
    "flag_worst_regime_drawdown_below_minus_20pct": bool(worst_regime_dd < -0.20) if np.isfinite(worst_regime_dd) else False,
}])

governance_flags["review_required"] = governance_flags[
    [
        "flag_top_model_too_dominant",
        "flag_top_model_low_positive_alpha_probability",
        "flag_effective_n_models_below_2",
        "flag_predictive_interval_miscalibrated",
        "flag_bma_test_sharpe_negative",
        "flag_worst_regime_drawdown_below_minus_20pct",
    ]
].any(axis=1)

governance_flags

## 24. Audit checks

Numerical and process checks:

1. posterior probabilities sum to one;
2. posterior samples are finite;
3. train/validation/test are chronological;
4. model weights are non-negative;
5. Bayesian model average has finite returns;
6. posterior predictive interval is ordered;
7. walk-forward posterior weights sum to one in each split.

In [ ]:
audit_rows = []

prob_sum = prob_series.sum()
audit_rows.append({
    "check": "posterior_model_probabilities_sum_to_one",
    "value": float(prob_sum),
    "passed": bool(abs(prob_sum - 1.0) < 1e-10),
})

samples_finite = True
for samples in posterior_sample_store.values():
    samples_finite = samples_finite and np.isfinite(samples["mu"]).all() and np.isfinite(samples["sigma"]).all() and np.isfinite(samples["sharpe"]).all()

audit_rows.append({
    "check": "posterior_samples_finite",
    "value": bool(samples_finite),
    "passed": bool(samples_finite),
})

chronological = train_returns.index[-1] < validation_returns.index[0] and validation_returns.index[-1] < test_returns.index[0]
audit_rows.append({
    "check": "train_validation_test_chronological",
    "value": bool(chronological),
    "passed": bool(chronological),
})

model_weights_nonnegative = bool((model_weights["posterior_model_probability"] >= -1e-12).all())
audit_rows.append({
    "check": "model_weights_nonnegative",
    "value": model_weights_nonnegative,
    "passed": model_weights_nonnegative,
})

bma_finite = bool(np.isfinite(comparison_returns["bayesian_model_average"]).all())
audit_rows.append({
    "check": "bayesian_model_average_returns_finite",
    "value": bma_finite,
    "passed": bma_finite,
})

interval_ordered = interval["p01"] <= interval["p05"] <= interval["p50"] <= interval["p95"] <= interval["p99"]
audit_rows.append({
    "check": "posterior_predictive_interval_ordered",
    "value": bool(interval_ordered),
    "passed": bool(interval_ordered),
})

wf_prob_sums = wf_model_weights.groupby("split_id")["posterior_model_probability"].sum()
wf_probs_sum_to_one = bool(np.allclose(wf_prob_sums.values, 1.0))
audit_rows.append({
    "check": "walk_forward_model_weights_sum_to_one",
    "value": float(np.max(np.abs(wf_prob_sums.values - 1.0))),
    "passed": wf_probs_sum_to_one,
})

audit_report = pd.DataFrame(audit_rows)

audit_report

## 25. Practical checklist for Bayesian calibration

Before trusting Bayesian strategy calibration:

1. **Define the prior explicitly**  
   A hidden prior is still a prior.

2. **Use conservative alpha priors**  
   Most strategies do not have large persistent alpha.

3. **Shrink noisy estimates**  
   High Sharpe with few observations should not dominate.

4. **Report posterior probabilities**  
   Avoid pretending the best model is certain.

5. **Use posterior predictive checks**  
   The model should forecast uncertainty realistically.

6. **Prefer model averaging when uncertainty is high**  
   One winner can be fragile.

7. **Walk forward**  
   Posterior updating must be chronological.

8. **Check stress regimes**  
   A posterior average can still fail in crises.

9. **Compare to point-estimate selection**  
   Bayesian calibration should reduce overconfidence.

10. **Save all posterior assumptions**  
   Priors, likelihood, samples, and model weights must be auditable.

## 26. Save outputs and manifest

In [ ]:
output_dir = Path("data/processed/bayesian_strategy_calibration_v1")
output_dir.mkdir(parents=True, exist_ok=True)

paths = {
    "returns": output_dir / "returns.csv",
    "prices": output_dir / "prices.csv",
    "metadata": output_dir / "metadata.csv",
    "factors": output_dir / "factors.csv",
    "regime_info": output_dir / "regime_info.csv",
    "benchmark": output_dir / "benchmark.csv",
    "parameter_grid": output_dir / "parameter_grid.csv",
    "strategy_returns": output_dir / "strategy_returns.csv",
    "split_report": output_dir / "split_report.csv",
    "point_estimates_train": output_dir / "point_estimates_train.csv",
    "posterior_table": output_dir / "posterior_table.csv",
    "posterior_summary": output_dir / "posterior_summary.csv",
    "shrinkage_comparison": output_dir / "shrinkage_comparison.csv",
    "model_weights": output_dir / "posterior_model_weights.csv",
    "comparison_returns": output_dir / "comparison_returns.csv",
    "bayesian_choice_performance": output_dir / "bayesian_choice_performance.csv",
    "predictive_summary": output_dir / "posterior_predictive_summary.csv",
    "thompson_selection_frequency": output_dir / "thompson_selection_frequency.csv",
    "wf_splits": output_dir / "walk_forward_splits.csv",
    "wf_bayesian_returns": output_dir / "walk_forward_bayesian_returns.csv",
    "wf_model_weights": output_dir / "walk_forward_model_weights.csv",
    "wf_selected_summary": output_dir / "walk_forward_selected_summary.csv",
    "wf_direct_performance": output_dir / "walk_forward_direct_performance.csv",
    "bayesian_regime_perf": output_dir / "bayesian_regime_performance.csv",
    "calibration_report": output_dir / "calibration_report.csv",
    "governance_flags": output_dir / "governance_flags.csv",
    "audit_report": output_dir / "audit_report.csv",
    "manifest": output_dir / "manifest.json",
}

returns.to_csv(paths["returns"])
prices.to_csv(paths["prices"])
metadata.to_csv(paths["metadata"], index=False)
factors.to_csv(paths["factors"])
regime_info.to_csv(paths["regime_info"])
benchmark.to_frame("benchmark_return").to_csv(paths["benchmark"])
pd.DataFrame(param_grid).assign(param_id=[params_to_id(p) for p in param_grid]).to_csv(paths["parameter_grid"], index=False)
strategy_returns.to_csv(paths["strategy_returns"])
split_report.to_frame("value").to_csv(paths["split_report"])
point_estimates_train.to_csv(paths["point_estimates_train"], index=False)
posterior_table.to_csv(paths["posterior_table"], index=False)
posterior_summary.to_csv(paths["posterior_summary"], index=False)
shrinkage_comparison.to_csv(paths["shrinkage_comparison"], index=False)
model_weights.to_csv(paths["model_weights"], index=False)
comparison_returns.to_csv(paths["comparison_returns"])
bayesian_choice_performance.to_csv(paths["bayesian_choice_performance"], index=False)
predictive_summary.to_csv(paths["predictive_summary"], index=False)
thompson_selection_frequency.to_csv(paths["thompson_selection_frequency"], index=False)
wf_splits.to_csv(paths["wf_splits"], index=False)
wf_bayesian_returns.to_csv(paths["wf_bayesian_returns"])
wf_model_weights.to_csv(paths["wf_model_weights"], index=False)
wf_selected_summary.to_csv(paths["wf_selected_summary"], index=False)
wf_direct_performance.to_csv(paths["wf_direct_performance"], index=False)
bayesian_regime_perf.to_csv(paths["bayesian_regime_perf"], index=False)
calibration_report.to_csv(paths["calibration_report"], index=False)
governance_flags.to_csv(paths["governance_flags"], index=False)
audit_report.to_csv(paths["audit_report"], index=False)

manifest = {
    "dataset_name": "bayesian_strategy_calibration_outputs",
    "schema_version": "bayesian_strategy_calibration_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "n_parameter_sets": len(param_grid),
    "top_posterior_param": top_selected_param,
    "point_estimate_best_param": point_best_param,
    "effective_n_models": float(effective_n_models),
    "bayesian_choice_performance": bayesian_choice_performance.to_dict(orient="records"),
    "walk_forward_direct_performance": wf_direct_performance.to_dict(orient="records"),
    "governance_flags": governance_flags.to_dict(orient="records"),
    "audit_passed": bool(audit_report["passed"].all()),
    "notes": [
        "The likelihood assumes daily strategy returns are conditionally normal.",
        "The prior is Normal-Inverse-Gamma over mean and variance.",
        "Posterior alpha and Sharpe are estimated by Monte Carlo sampling.",
        "Posterior model probabilities are softmax weights over uncertainty-adjusted model scores.",
        "Bayesian model averaging combines strategy returns using posterior model weights.",
        "Walk-forward Bayesian updating recalibrates posterior model weights using only trailing data.",
        "This notebook is about uncertainty-aware calibration, not proof of strategy profitability."
    ],
}

paths["manifest"].write_text(json.dumps(manifest, indent=2, default=str))

paths["posterior_summary"], paths["model_weights"], paths["wf_direct_performance"], paths["governance_flags"], paths["manifest"]

## 27. Limitations

### Normal return likelihood

The model assumes approximately normal daily strategy returns. Real strategies can have skew, jumps, serial correlation, and volatility clustering.

### Simple priors

The prior is deliberately transparent but not necessarily optimal.

### Softmax model probabilities

The model probabilities are practical scoring weights, not exact marginal likelihoods.

### No transaction-cost uncertainty

Costs are fixed in the backtest. A full Bayesian model would include uncertainty over costs and slippage.

### No structural regime posterior

Regime information is analysed after the fact, but not modelled explicitly in the Bayesian posterior.

### Parameter grid is finite

The calibration only compares parameter sets in the chosen grid.

### Posterior predictive simplification

Predictive simulations assume independent returns, which ignores autocorrelation and path-dependent risk.

## 28. Research frontier and extensions

Important extensions include:

1. Student-t likelihoods for fat-tailed strategy returns;
2. Bayesian hierarchical shrinkage across similar parameter sets;
3. Gaussian-process Bayesian optimisation;
4. Bayesian regime-switching strategy calibration;
5. posterior over transaction-cost assumptions;
6. Bayesian model averaging with exact marginal likelihoods;
7. dynamic linear models for time-varying alpha;
8. particle filtering for online parameter learning;
9. Bayesian portfolio allocation from posterior predictive returns;
10. Chinese futures Bayesian calibration with contract-specific liquidity and regime states.

## 29. Suggested follow-up notebooks

This notebook naturally leads to:

1. **05_06_walk_forward_model_validation**  
   Validate predictive ML models chronologically.

2. **05_07_parameter_sensitivity_and_overfitting**  
   Compare Bayesian uncertainty with overfitting diagnostics.

3. **05_08_strategy_capacity_and_market_impact**  
   Add posterior uncertainty over capacity and costs.

4. **05_10_research_audit_trail_and_manifest**  
   Save priors, posterior samples, configs and hashes.

5. **05_11_live_shadow_mode_monitoring**  
   Compare posterior expected performance with paper/live outcomes.

6. **07_05_bayesian_alpha_research_capstone**  
   Full Bayesian research process from signal to allocation.

## 30. Summary

This notebook implemented Bayesian strategy calibration.

It showed how to:

1. simulate a multi-asset research dataset;
2. generate a parameterised strategy family;
3. backtest every parameter set after costs;
4. compute frequentist point estimates;
5. fit Normal-Inverse-Gamma posteriors;
6. sample posterior mean, volatility and Sharpe;
7. estimate probability of positive alpha;
8. estimate probability Sharpe exceeds a threshold;
9. compare posterior shrinkage to point estimates;
10. compute posterior model probabilities;
11. build Bayesian model averaging returns;
12. compare Bayesian choices on validation and test periods;
13. simulate posterior predictive performance;
14. implement Thompson-style parameter sampling;
15. run walk-forward Bayesian updating;
16. diagnose posterior concentration;
17. check regime-conditioned performance;
18. check predictive interval calibration;
19. create governance flags and audit checks;
20. save outputs and manifests.

The key computational takeaway:

> Bayesian calibration turns noisy strategy returns into posterior distributions over alpha, volatility, Sharpe and model probability.

The key financial takeaway:

> The best-looking parameter set is often just the loudest noise; Bayesian calibration forces it to earn posterior credibility.

## 31. Further reading

- Gelman et al., *Bayesian Data Analysis.*
- Murphy, *Machine Learning: A Probabilistic Perspective.*
- López de Prado, *Advances in Financial Machine Learning.*
- Bailey et al. on backtest overfitting.
- Grinold and Kahn, *Active Portfolio Management.*
- Literature on Bayesian model averaging, Bayesian optimisation, dynamic linear models and probabilistic forecasting.